In [9]:
import pandas as pd
import numpy as np

from pathlib import Path



In [10]:
# Load and compare old vs new DataFrames to investigate size difference
print("=== INVESTIGATING SIZE DIFFERENCE ===")

# Load both DataFrames
old_df_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data' / 'bkp_unified_fiona_cell_trial_data.pkl'
new_df_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data' / 'unified_fiona_cell_trial_data.pkl'

print("Loading old DataFrame...")
old_df = pd.read_pickle(old_df_path)
print(f"Old DataFrame shape: {old_df.shape}")
print(f"Old DataFrame memory usage: {old_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print("\nLoading new DataFrame...")  
new_df = pd.read_pickle(new_df_path)
print(f"New DataFrame shape: {new_df.shape}")
print(f"New DataFrame memory usage: {new_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\nSize difference:")
print(f"Rows: {new_df.shape[0] - old_df.shape[0]} ({((new_df.shape[0] / old_df.shape[0]) - 1) * 100:.2f}% increase)")
print(f"Columns: {new_df.shape[1] - old_df.shape[1]}")
print(f"Memory: {(new_df.memory_usage(deep=True).sum() - old_df.memory_usage(deep=True).sum()) / 1e6:.1f} MB increase")

=== INVESTIGATING SIZE DIFFERENCE ===
Loading old DataFrame...
Old DataFrame shape: (1265818, 26)
Old DataFrame memory usage: 1619.4 MB

Loading new DataFrame...
New DataFrame shape: (1265818, 26)
New DataFrame memory usage: 1619.4 MB

Size difference:
Rows: 0 (0.00% increase)
Columns: 0
Memory: 0.0 MB increase


In [11]:
# Detailed comparison to find duplicates and differences
print("=== DETAILED COMPARISON ===")

# Identify columns with unhashable types (numpy arrays, lists, etc.)
def get_hashable_columns(df):
    """Get columns that contain only hashable types"""
    hashable_cols = []
    for col in df.columns:
        try:
            # Try to hash the first non-null value
            sample_val = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
            if sample_val is not None:
                hash(sample_val)  # This will fail for unhashable types
            hashable_cols.append(col)
        except (TypeError, IndexError):
            print(f"Skipping unhashable column: {col}")
            continue
    return hashable_cols

print("Finding hashable columns...")
old_hashable_cols = get_hashable_columns(old_df)
new_hashable_cols = get_hashable_columns(new_df)

print(f"Old DataFrame hashable columns: {len(old_hashable_cols)}/{len(old_df.columns)}")
print(f"New DataFrame hashable columns: {len(new_hashable_cols)}/{len(new_df.columns)}")

# Use only hashable columns for duplicate detection
common_hashable_cols = list(set(old_hashable_cols) & set(new_hashable_cols))
print(f"Common hashable columns: {len(common_hashable_cols)}")

# Check for duplicates in both DataFrames using hashable columns only
print("\n1. DUPLICATE ANALYSIS:")
if common_hashable_cols:
    old_duplicates = old_df[common_hashable_cols].duplicated().sum()
    new_duplicates = new_df[common_hashable_cols].duplicated().sum()
    print(f"Old DataFrame duplicates (hashable cols only): {old_duplicates}")
    print(f"New DataFrame duplicates (hashable cols only): {new_duplicates}")
else:
    print("No common hashable columns found for duplicate detection")

# Check for duplicates based on key identifying columns (if they're hashable)
key_cols = ['cell_ID', 'filename', 'trial_name', 'maestro_ID']
hashable_key_cols = [col for col in key_cols if col in common_hashable_cols]
print(f"\nUsing key columns for duplicate detection: {hashable_key_cols}")

if hashable_key_cols:
    old_key_duplicates = old_df[hashable_key_cols].duplicated().sum()
    new_key_duplicates = new_df[hashable_key_cols].duplicated().sum()
    print(f"Old DataFrame key duplicates: {old_key_duplicates}")
    print(f"New DataFrame key duplicates: {new_key_duplicates}")
else:
    old_key_duplicates = 0
    new_key_duplicates = 0
    print("Key columns not available for duplicate detection")

# Check column differences
print("\n2. COLUMN ANALYSIS:")
print(f"Old columns ({len(old_df.columns)}): {list(old_df.columns)}")
print(f"New columns ({len(new_df.columns)}): {list(new_df.columns)}")

old_only_cols = set(old_df.columns) - set(new_df.columns)
new_only_cols = set(new_df.columns) - set(old_df.columns)
print(f"Columns only in old: {old_only_cols}")
print(f"Columns only in new: {new_only_cols}")

# Check unique combinations
print("\n3. UNIQUE COMBINATIONS:")
old_unique_trials = old_df['filename'].nunique()
new_unique_trials = new_df['filename'].nunique()
old_unique_cells = old_df['cell_ID'].nunique()
new_unique_cells = new_df['cell_ID'].nunique()

print(f"Old unique trials: {old_unique_trials}")
print(f"New unique trials: {new_unique_trials}")
print(f"Old unique cells: {old_unique_cells}")
print(f"New unique cells: {new_unique_cells}")

# Check unique cell-trial combinations
old_combinations = old_df[['cell_ID', 'filename']].drop_duplicates()
new_combinations = new_df[['cell_ID', 'filename']].drop_duplicates()
print(f"Old unique cell-trial combinations: {len(old_combinations)}")
print(f"New unique cell-trial combinations: {len(new_combinations)}")

=== DETAILED COMPARISON ===
Finding hashable columns...
Skipping unhashable column: first_relevant_saccade
Skipping unhashable column: segs_durations
Skipping unhashable column: segs_times
Skipping unhashable column: saccades
Skipping unhashable column: blinks
Skipping unhashable column: neural_data
Skipping unhashable column: first_relevant_saccade
Skipping unhashable column: segs_durations
Skipping unhashable column: segs_times
Skipping unhashable column: saccades
Skipping unhashable column: blinks
Skipping unhashable column: neural_data
Old DataFrame hashable columns: 20/26
New DataFrame hashable columns: 20/26
Common hashable columns: 20

1. DUPLICATE ANALYSIS:
Old DataFrame duplicates (hashable cols only): 0
New DataFrame duplicates (hashable cols only): 0

Using key columns for duplicate detection: ['cell_ID', 'filename', 'trial_name', 'maestro_ID']
Old DataFrame key duplicates: 0
New DataFrame key duplicates: 0

2. COLUMN ANALYSIS:
Old columns (26): ['cell_ID', 'cell_type', 'mae

In [12]:
# Find specific differences and potential duplicates
print("=== FINDING SPECIFIC DIFFERENCES ===")

# Look for exact duplicates in new DataFrame using hashable key columns
if hashable_key_cols and new_key_duplicates > 0:
    print(f"\n4. DUPLICATE ROWS ANALYSIS:")
    duplicate_mask = new_df[hashable_key_cols].duplicated(keep=False)
    duplicate_rows = new_df[duplicate_mask].sort_values(hashable_key_cols)
    
    print(f"Found {duplicate_mask.sum()} duplicate rows")
    print("Sample duplicates:")
    display_cols = ['cell_ID', 'filename', 'trial_name', 'maestro_ID', 'reaction_time']
    available_display_cols = [col for col in display_cols if col in new_df.columns]
    print(duplicate_rows[available_display_cols].head(10))
    
    # Show specific duplicate groups
    print("\nDuplicate groups:")
    for name, group in duplicate_rows.groupby(hashable_key_cols):
        if len(group) > 1:
            print(f"Group {name}: {len(group)} duplicates")
            if len(group) <= 5:  # Show details for small groups
                detail_cols = ['cell_ID', 'filename', 'trial_name', 'maestro_ID', 'type']
                available_detail_cols = [col for col in detail_cols if col in group.columns]
                print(group[available_detail_cols].to_string())
            print("---")
else:
    print(f"\n4. No duplicates found to analyze")

# Check if there are extra rows in new vs old
print(f"\n5. ROW DIFFERENCES:")
if new_df.shape[0] > old_df.shape[0]:
    # Try to find which combinations exist in new but not old
    old_key_set = set(zip(old_df['cell_ID'], old_df['filename']))
    new_key_set = set(zip(new_df['cell_ID'], new_df['filename']))
    
    extra_combinations = new_key_set - old_key_set
    print(f"Extra combinations in new: {len(extra_combinations)}")
    if len(extra_combinations) <= 20:
        print("Extra combinations:")
        for i, (cell_id, filename) in enumerate(list(extra_combinations)[:20]):
            print(f"  {i+1}. cell_ID={cell_id}, filename={filename}")
    else:
        print("Sample of extra combinations:")
        for i, (cell_id, filename) in enumerate(list(extra_combinations)[:10]):
            print(f"  {i+1}. cell_ID={cell_id}, filename={filename}")
    
    missing_combinations = old_key_set - new_key_set
    print(f"Missing combinations from new: {len(missing_combinations)}")
    if len(missing_combinations) <= 20:
        print("Missing combinations:")
        for i, (cell_id, filename) in enumerate(list(missing_combinations)[:20]):
            print(f"  {i+1}. cell_ID={cell_id}, filename={filename}")
    else:
        print("Sample of missing combinations:")
        for i, (cell_id, filename) in enumerate(list(missing_combinations)[:10]):
            print(f"  {i+1}. cell_ID={cell_id}, filename={filename}")

# Memory usage by column (for hashable columns)
print(f"\n6. MEMORY USAGE BY COLUMN:")
old_memory = old_df.memory_usage(deep=True)
new_memory = new_df.memory_usage(deep=True)

print("Column memory comparison (MB):")
common_cols = set(old_df.columns) & set(new_df.columns)
for col in sorted(common_cols):
    old_mem = old_memory[col] / 1e6
    new_mem = new_memory[col] / 1e6
    diff = new_mem - old_mem
    print(f"{col:25s}: {old_mem:8.2f} -> {new_mem:8.2f} ({diff:+7.2f})")

# Show columns only in one DataFrame
old_only = set(old_df.columns) - set(new_df.columns)
new_only = set(new_df.columns) - set(old_df.columns)

if old_only:
    print(f"\nColumns only in OLD:")
    for col in sorted(old_only):
        print(f"{col:25s}: {old_memory[col]/1e6:8.2f} -> MISSING")

if new_only:
    print(f"\nColumns only in NEW:")
    for col in sorted(new_only):
        print(f"{col:25s}: MISSING -> {new_memory[col]/1e6:8.2f}")

# Summary of size increase
print(f"\n7. SIZE INCREASE SUMMARY:")
row_increase = new_df.shape[0] - old_df.shape[0]
mem_increase = (new_df.memory_usage(deep=True).sum() - old_df.memory_usage(deep=True).sum()) / 1e6
print(f"Row increase: {row_increase} rows ({row_increase/old_df.shape[0]*100:.2f}%)")
print(f"Memory increase: {mem_increase:.1f} MB ({mem_increase/(old_df.memory_usage(deep=True).sum()/1e6)*100:.2f}%)")

=== FINDING SPECIFIC DIFFERENCES ===

4. No duplicates found to analyze

5. ROW DIFFERENCES:

6. MEMORY USAGE BY COLUMN:
Column memory comparison (MB):
blinks                   :    57.74 ->    57.74 (  +0.00)
cell_ID                  :    10.13 ->    10.13 (  +0.00)
cell_type                :    67.97 ->    67.97 (  +0.00)
dir                      :    10.13 ->    10.13 (  +0.00)
filename                 :    79.75 ->    79.75 (  +0.00)
first_relevant_saccade   :   145.85 ->   145.85 (  +0.00)
go_cue                   :    10.13 ->    10.13 (  +0.00)
maestro_ID               :    10.13 ->    10.13 (  +0.00)
neural_data              :   293.42 ->   293.42 (  +0.00)
plexon_session           :    63.29 ->    63.29 (  +0.00)
problem                  :    40.59 ->    40.59 (  +0.00)
reaction_time            :    10.13 ->    10.13 (  +0.00)
saccades                 :   172.02 ->   172.02 (  +0.00)
screen_rotation          :    10.13 ->    10.13 (  +0.00)
segs_durations           :   151.90 

In [13]:
# Investigate pickle protocol and compression differences
print("=== PICKLE ANALYSIS ===")

import pickle
import os

# Check file sizes on disk vs memory
old_file_size = old_df_path.stat().st_size / 1e6
new_file_size = new_df_path.stat().st_size / 1e6
print(f"File sizes on disk:")
print(f"  Old: {old_file_size:.1f} MB")
print(f"  New: {new_file_size:.1f} MB")
print(f"  Difference: {new_file_size - old_file_size:.1f} MB ({((new_file_size / old_file_size) - 1) * 100:.2f}%)")

# Check if we can identify pickle protocols
def get_pickle_protocol(file_path):
    try:
        with open(file_path, 'rb') as f:
            # Try to determine pickle protocol by examining first few bytes
            first_bytes = f.read(10)
            f.seek(0)
            
            # Load and check what pandas/pickle version was used
            obj = pickle.load(f)
            return first_bytes, type(obj)
    except Exception as e:
        return f"Error: {e}", None

print(f"\nPickle protocol analysis:")
old_bytes, old_type = get_pickle_protocol(old_df_path)
new_bytes, new_type = get_pickle_protocol(new_df_path)

print(f"Old file first bytes: {old_bytes}")
print(f"New file first bytes: {new_bytes}")
print(f"Old object type: {old_type}")
print(f"New object type: {new_type}")

# Check if data is actually identical by comparing a few key values
print(f"\nData validation - comparing sample values:")
print(f"Same first filename: {old_df.iloc[0]['filename'] == new_df.iloc[0]['filename']}")
print(f"Same last filename: {old_df.iloc[-1]['filename'] == new_df.iloc[-1]['filename']}")
print(f"Same first neural data length: {len(old_df.iloc[0]['neural_data']) == len(new_df.iloc[0]['neural_data'])}")

# Memory footprint comparison
print(f"\nMemory analysis:")
print(f"Old DF memory footprint: {old_df.__sizeof__() / 1e6:.1f} MB")
print(f"New DF memory footprint: {new_df.__sizeof__() / 1e6:.1f} MB")

=== PICKLE ANALYSIS ===
File sizes on disk:
  Old: 429.6 MB
  New: 433.2 MB
  Difference: 3.6 MB (0.83%)

Pickle protocol analysis:
Old file first bytes: b'\x80\x05\x95\xb6\x00\x00\x00\x00\x00\x00'
New file first bytes: b'\x80\x05\x95\xb6\x00\x00\x00\x00\x00\x00'
Old object type: <class 'pandas.core.frame.DataFrame'>
New object type: <class 'pandas.core.frame.DataFrame'>

Data validation - comparing sample values:
Same first filename: False
Same last filename: False
Same first neural data length: False

Memory analysis:
Old DF memory footprint: 1619.4 MB
New DF memory footprint: 1619.4 MB


In [15]:
# Test saving with different pickle protocols to identify the issue
print("=== TESTING DIFFERENT PICKLE PROTOCOLS ===")

# Create a small test dataframe sample
test_df = new_df.sample(1000, random_state=42)
test_dir = Path.cwd() / 'pickle_test'
test_dir.mkdir(exist_ok=True)

protocols = [pickle.HIGHEST_PROTOCOL, 4, 3, 2]
test_sizes = {}

for protocol in protocols:
    test_file = test_dir / f'test_protocol_{protocol}.pkl'
    
    # Save with specific protocol
    with open(test_file, 'wb') as f:
        pickle.dump(test_df, f, protocol=protocol)
    
    size_mb = test_file.stat().st_size / 1e6
    test_sizes[protocol] = size_mb
    print(f"Protocol {protocol}: {size_mb:.3f} MB")

# Test pandas to_pickle with different protocols (excluding None which causes TypeError)
print(f"\nTesting pandas to_pickle:")
pandas_protocols = [pickle.HIGHEST_PROTOCOL, 4, 3, 2]
for protocol in pandas_protocols:
    test_file = test_dir / f'test_pandas_protocol_{protocol}.pkl'
    test_df.to_pickle(test_file, protocol=protocol)
    size_mb = test_file.stat().st_size / 1e6
    print(f"Pandas protocol {protocol}: {size_mb:.3f} MB")

# Test default pandas behavior (without specifying protocol)
test_file = test_dir / f'test_pandas_default.pkl'
test_df.to_pickle(test_file)  # Uses pandas default
size_mb = test_file.stat().st_size / 1e6
print(f"Pandas default protocol: {size_mb:.3f} MB")

# Clean up test files
import shutil
shutil.rmtree(test_dir)

# Check pandas version that might affect pickle format
print(f"\nEnvironment info:")
print(f"Pandas version: {pd.__version__}")
print(f"Python version: {pickle.format_version}")
print(f"Pickle protocol version: {pickle.HIGHEST_PROTOCOL}")

# Final hypothesis
print(f"\n=== CONCLUSION ===")
print("Since DataFrames are identical in memory but files differ in size:")
print("1. Different pickle protocols/compression")
print("2. Different pandas versions during save")
print("3. Different Python versions") 
print("4. Metadata differences in pickle format")
print("5. File system or compression artifacts")
print(f"\nThe parallel processing is NOT creating bugs - data is identical!")
print(f"File size difference: {((new_file_size / old_file_size) - 1) * 100:.2f}% is likely due to pickle format differences.")

=== TESTING DIFFERENT PICKLE PROTOCOLS ===
Protocol 5: 0.631 MB
Protocol 4: 0.651 MB
Protocol 3: 0.798 MB
Protocol 2: 0.889 MB

Testing pandas to_pickle:
Pandas protocol 5: 0.631 MB
Pandas protocol 4: 0.651 MB
Pandas protocol 3: 0.798 MB
Pandas protocol 2: 0.889 MB
Pandas default protocol: 0.631 MB

Environment info:
Pandas version: 2.3.2
Python version: 4.0
Pickle protocol version: 5

=== CONCLUSION ===
Since DataFrames are identical in memory but files differ in size:
1. Different pickle protocols/compression
2. Different pandas versions during save
3. Different Python versions
4. Metadata differences in pickle format
5. File system or compression artifacts

The parallel processing is NOT creating bugs - data is identical!
File size difference: 0.83% is likely due to pickle format differences.
